# AITR Lab 02 — Exploration Notebook
## MRA14 | LLM Performance Drift: Embedding-Space vs. Accuracy-Based Detection

This notebook walks through the entire study interactively — suitable for the 5-minute demonstration (Gate G4).

**Run order:** cell by cell from top to bottom.
For the full experiment, prefer the scripts in `scripts/` which are more efficient.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')  # project root

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
with open('../configs/experiment_config.json') as f:
    CFG = json.load(f)
print(json.dumps(CFG, indent=2))

## 1. Load Dataset & Inspect Drift Partitions

In [ ]:
from src.data_loader import load_triviaqa, partition_dataset, create_drift_stream

df = load_triviaqa(n_samples=CFG['data']['n_total'], seed=CFG['seed'])
print('Category distribution:')
print(df['category'].value_counts())

In [ ]:
reference, stable_stream, drift_stream = partition_dataset(
    df,
    n_reference=CFG['data']['n_reference'],
    n_stable_stream=CFG['data']['n_stable_stream'],
    n_drift_stream=CFG['data']['n_drift_stream'],
    seed=CFG['seed']
)

stream = create_drift_stream(stable_stream, drift_stream)
drift_onset = int(stream[stream['is_drift']].index.min())
print(f'\nTest stream length: {len(stream)}')
print(f'Drift onset at index: {drift_onset}')

In [ ]:
# Show 3 stable + 3 drift example questions
print('=== STABLE (in-distribution) examples ===')
for _, row in reference.head(3).iterrows():
    print(f'  Q: {row["question"]}')
    print(f'  A: {row["answers"][0]}\n')

print('=== DRIFT examples ===')
for _, row in drift_stream.head(3).iterrows():
    print(f'  Q: {row["question"]}')
    print(f'  A: {row["answers"][0]}\n')

## 2. Load Models & Run Mini Inference Test

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

print('Loading gpt2-medium...')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-medium')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2-medium').to(DEVICE)
model.eval()
print('Loaded.')

In [ ]:
def qa_inference(question, max_new_tokens=50):
    prompt = f'Q: {question}\nA:'
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model(**inputs)
        logits = output.logits[0, -1, :].cpu().numpy()  # first generation logit
        gen_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(
        gen_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    return answer, logits

# Quick test
test_q = 'What is the capital of France?'
ans, logits = qa_inference(test_q)
print(f'Q: {test_q}')
print(f'A (GPT-2-medium): {ans}')
print(f'Logit vector shape: {logits.shape}')

## 3. Sentence Embeddings (for MMD-MON)

In [ ]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(CFG['embedding_model'])
test_emb = emb_model.encode([test_q])[0]
print(f'Embedding shape: {test_emb.shape}')  # expected (384,)

# Visualise embeddings for 10 stable vs 10 drift questions
from sklearn.decomposition import PCA

stable_qs = reference['question'].head(50).tolist()
drift_qs  = drift_stream['question'].head(50).tolist()

stable_embs = emb_model.encode(stable_qs)
drift_embs  = emb_model.encode(drift_qs)

all_embs = np.vstack([stable_embs, drift_embs])
pca = PCA(n_components=2)
reduced = pca.fit_transform(all_embs)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(reduced[:50, 0], reduced[:50, 1], label='Stable', alpha=0.6, color='steelblue')
ax.scatter(reduced[50:, 0], reduced[50:, 1], label='Drift',  alpha=0.6, color='tomato')
ax.set_title('PCA of Sentence Embeddings: Stable vs Drift Questions')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/embedding_pca.png', dpi=150)
plt.show()
print('Saved embedding PCA figure.')

## 4. Drift Detector Unit Tests

In [ ]:
from src.drift_detector import AccuracyMonitor, JSDMonitor, MMDMonitor, _jsd, _mmd_squared

# --- JSD unit test ---
p = np.array([0.9, 0.05, 0.05])    # peaked distribution
q = np.array([0.33, 0.33, 0.34])   # uniform distribution
r = np.array([0.9, 0.05, 0.05])    # same as p

print(f'JSD(p, p) = {_jsd(p, r):.6f}  (expected ≈ 0.0)')
print(f'JSD(p, q) = {_jsd(p, q):.6f}  (expected > 0, ≤ ln2 ≈ 0.693)')
assert _jsd(p, r) < 1e-6, 'JSD(p,p) should be ~0'
assert 0 < _jsd(p, q) < 0.693, 'JSD should be in (0, ln2)'
print('✅ JSD unit tests passed.')

In [ ]:
# --- MMD unit test ---
rng = np.random.default_rng(42)
X_same = rng.normal(0, 1, (100, 10))
Y_same = rng.normal(0, 1, (100, 10))  # same distribution
Y_diff = rng.normal(5, 1, (100, 10))  # different distribution

mmd_same = _mmd_squared(X_same, Y_same)
mmd_diff = _mmd_squared(X_same, Y_diff)

print(f'MMD²(same distributions) = {mmd_same:.6f}  (expected ≈ 0)')
print(f'MMD²(different distributions) = {mmd_diff:.6f}  (expected > 0)')
assert mmd_same < 0.1, f'MMD² for same dist should be small, got {mmd_same}'
assert mmd_diff > mmd_same, 'MMD² for different dist should be larger'
print('✅ MMD unit tests passed.')

## 5. Mini Drift Experiment (100 samples)
A fast preview — the full experiment runs via `scripts/03_run_experiment.py`.

In [ ]:
from src.evaluator import compute_exact_match, compute_f1

# Use first 60 reference + first 20 stable + first 20 drift = 100 stream steps
MINI_REF    = reference.head(60)
MINI_STABLE = stable_stream.head(20)
MINI_DRIFT  = drift_stream.head(20)
MINI_STREAM = pd.concat([MINI_STABLE, MINI_DRIFT], ignore_index=True)
MINI_DRIFT_ONSET = len(MINI_STABLE)

print(f'Mini reference: {len(MINI_REF)} | Stream: {len(MINI_STREAM)} | Drift onset: {MINI_DRIFT_ONSET}')

In [ ]:
# Compute reference statistics
ref_logits_list = []
ref_embs_list   = []
ref_correctness = []

print('Computing reference statistics...')
for _, row in tqdm(MINI_REF.iterrows(), total=len(MINI_REF)):
    ans, logits = qa_inference(row['question'])
    emb = emb_model.encode([row['question']])[0]
    em  = compute_exact_match(ans, row['answers'])
    ref_logits_list.append(logits)
    ref_embs_list.append(emb)
    ref_correctness.append(em)

ref_logits = np.array(ref_logits_list)
ref_embs   = np.array(ref_embs_list)
print(f'Reference EM: {np.mean(ref_correctness)*100:.1f}%')

In [ ]:
# Initialise and calibrate monitors
MINI_WIN = 10  # small window for the mini experiment

acc_mon = AccuracyMonitor(window_size=MINI_WIN)
acc_mon.calibrate_from_reference(ref_correctness)

jsd_mon = JSDMonitor(window_size=MINI_WIN)
jsd_mon.calibrate_from_reference(list(ref_logits))

mmd_mon = MMDMonitor(window_size=MINI_WIN)
mmd_mon.calibrate_from_reference(ref_embs)

print(f'Thresholds — ACC drop: {acc_mon.threshold_drop:.4f} | '
      f'JSD: {jsd_mon.threshold:.4f} | MMD: {mmd_mon.threshold:.6f}')

In [ ]:
# Run mini stream
records = []
print('Running mini stream...')
for idx, row in tqdm(MINI_STREAM.iterrows(), total=len(MINI_STREAM)):
    ans, logits = qa_inference(row['question'])
    emb = emb_model.encode([row['question']])[0]
    em  = compute_exact_match(ans, row['answers'])
    f1  = compute_f1(ans, row['answers'])

    acc_stat, acc_flag = acc_mon.update(bool(em))
    jsd_stat, jsd_flag = jsd_mon.update(logits)
    mmd_stat, mmd_flag = mmd_mon.update(emb)

    records.append({
        'idx': idx, 'em': em, 'f1': f1,
        'is_drift': bool(row['is_drift']),
        'acc_stat': acc_stat, 'acc_flag': acc_flag,
        'jsd_stat': jsd_stat, 'jsd_flag': jsd_flag,
        'mmd_stat': mmd_stat, 'mmd_flag': mmd_flag,
    })

res_df = pd.DataFrame(records)
print('Done.')

In [ ]:
# --- Plot mini results ---
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

x = res_df['idx'].values

# Task accuracy
axes[0].bar(x, res_df['em'], color=['tomato' if d else 'steelblue' for d in res_df['is_drift']])
axes[0].axvline(MINI_DRIFT_ONSET - 0.5, color='red', linestyle='--', label='Drift onset')
axes[0].set_ylabel('Exact Match'); axes[0].legend()
axes[0].set_title('Task Accuracy per Sample')

# ACC-MON
axes[1].plot(x, res_df['acc_stat'], color='#e74c3c', linewidth=1.5)
axes[1].axhline(acc_mon.threshold_drop, color='#e74c3c', linestyle=':', label='Threshold')
axes[1].axvline(MINI_DRIFT_ONSET - 0.5, color='red', linestyle='--')
axes[1].set_ylabel('ACC-MON\n(accuracy drop)'); axes[1].legend()

# JSD-MON
axes[2].plot(x, res_df['jsd_stat'], color='#f39c12', linewidth=1.5)
axes[2].axhline(jsd_mon.threshold, color='#f39c12', linestyle=':', label='Threshold')
axes[2].axvline(MINI_DRIFT_ONSET - 0.5, color='red', linestyle='--')
axes[2].set_ylabel('JSD-MON'); axes[2].legend()

# MMD-MON
axes[3].plot(x, res_df['mmd_stat'], color='#2980b9', linewidth=1.5)
axes[3].axhline(mmd_mon.threshold, color='#2980b9', linestyle=':', label='Threshold')
axes[3].axvline(MINI_DRIFT_ONSET - 0.5, color='red', linestyle='--')
axes[3].set_ylabel('MMD-MON\n(MMD²)'); axes[3].legend()
axes[3].set_xlabel('Stream index')

fig.suptitle('Mini Drift Experiment — GPT-2-medium (100 samples)', fontsize=13)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/mini_experiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mini experiment figure saved.')

In [ ]:
# Summary table for mini experiment
in_d = res_df[~res_df['is_drift']]
drft = res_df[res_df['is_drift']]

print('=== Mini Experiment Summary ===')
print(f'In-dist EM : {in_d["em"].mean()*100:.1f}%  |  F1: {in_d["f1"].mean()*100:.1f}%')
print(f'Drift   EM : {drft["em"].mean()*100:.1f}%  |  F1: {drft["f1"].mean()*100:.1f}%')
print()
for monitor, flag_col in [('ACC-MON','acc_flag'),('JSD-MON','jsd_flag'),('MMD-MON','mmd_flag')]:
    flags = res_df[flag_col].values
    post  = np.where(flags & (res_df['idx'] >= MINI_DRIFT_ONSET))[0]
    delay = int(post[0]) - MINI_DRIFT_ONSET if len(post) else 'Never'
    fp    = int((flags[:MINI_DRIFT_ONSET]).sum())
    print(f'{monitor}: first post-drift flag at delay={delay}, false positives pre-drift={fp}')

## 6. For the Full Experiment

Run the scripts directory for the full 700-sample stream and all three metrics:

```bash
python scripts/02_run_baseline.py
python scripts/03_run_experiment.py
python scripts/04_evaluate_results.py
python scripts/03_run_experiment.py --ablation
python scripts/04_evaluate_results.py --table 3
```

Full results (Tables 1–3) will be saved as CSV in `results/`.  
Figures will be saved in `results/figures/`.  
Copy PDFs to `paper/figures/` and compile `paper/main.tex`.